In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ===== 颜色定义 =====
COLOR_RED = "#E89B9B"      # 浅红色（seed_area）
COLOR_BLUE = "#7FACCF"     # 灰蓝色（leaf_area）
COLOR_GREEN = "#9EC29E"    # 浅绿色（root_area）
COLOR_GRAY = "#F0F0F0"     # 浅灰色（matched_dist）

# ===== matplotlib全局参数（符合规范）=====
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['axes.labelweight'] = 'normal'
plt.rcParams['axes.titleweight'] = 'normal'
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5

def get_area(points):
    x = [p[0] for p in points]
    y = [p[1] for p in points]
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def get_centroid(points):
    return np.mean(points, axis=0)

def main():
    raw_input = input("请输入包含 match_json 的文件夹路径: ").strip()
    clean_path = raw_input.strip('"').strip("'").replace('\\', '/')
    root_path = Path(clean_path).resolve()
    
    if not root_path.exists():
        print(f"错误：路径不存在 -> {root_path}")
        return
    
    metrics = {"seed_area": [], "root_area": [], "leaf_area": [], "matched_dist": []}
    json_files = list(root_path.glob("*.json"))
    print(f"找到 {len(json_files)} 个 JSON 文件，开始扫描...")

    found_any_gid = False
    for j_path in json_files:
        with open(j_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
                shapes = data.get('shapes', [])
                slots = {}
                for s in shapes:
                    gid = s.get('group_id')
                    if gid is None:
                        continue
                    gid = str(gid)
                    found_any_gid = True
                    label = s['label'].lower().strip()
                    pts = s['points']
                    if len(pts) < 3:
                        continue
                    area = get_area(pts)
                    centroid = get_centroid(pts)
                    if gid not in slots:
                        slots[gid] = {}
                    slots[gid][label] = {"area": area, "centroid": centroid}
                    if 'seed' in label:
                        metrics["seed_area"].append(area)
                    if 'root' in label:
                        metrics["root_area"].append(area)
                    if 'leaf' in label:
                        metrics["leaf_area"].append(area)
                for gid, parts in slots.items():
                    if 'seed' in parts and 'leaf' in parts:
                        d = np.linalg.norm(parts['seed']['centroid'] - parts['leaf']['centroid'])
                        metrics["matched_dist"].append(d)
            except Exception as e:
                print(f"处理文件 {j_path.name} 时出错: {e}")

    if not found_any_gid:
        print("警告：未检测到 group_id，请先运行个体匹配脚本。")

    # 输出统计量（辅助设定阈值）
    for key, vals in metrics.items():
        if vals:
            print(f"\n{key}:")
            print(f"  样本数 = {len(vals)}")
            print(f"  最小值 = {np.min(vals):.2f}")
            print(f"  25%分位数 = {np.percentile(vals, 25):.2f}")
            print(f"  50%分位数 = {np.percentile(vals, 50):.2f}")
            print(f"  75%分位数 = {np.percentile(vals, 75):.2f}")
            print(f"  最大值 = {np.max(vals):.2f}")
        else:
            print(f"\n{key}: 无数据")

    # 绘图（无 Panel 标签，无参考线）
    fig, axs = plt.subplots(2, 2, figsize=(10, 8), dpi=300)
    configs = [
        ("seed_area", "Seed Area", COLOR_RED, "Area (pixels²)"),
        ("root_area", "Root Area", COLOR_GREEN, "Area (pixels²)"),
        ("leaf_area", "Leaf Area", COLOR_BLUE, "Area (pixels²)"),
        ("matched_dist", "Seed-Leaf Distance", COLOR_GRAY, "Distance (pixels)")
    ]
    for i, (key, title, color, xlabel) in enumerate(configs):
        ax = axs.flat[i]
        vals = metrics[key]
        if vals:
            ax.hist(vals, bins=40, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
            ax.set_title(title, fontsize=12, fontweight='normal')
            ax.set_xlabel(xlabel, fontsize=11, fontweight='normal')
            ax.set_ylabel('Frequency', fontsize=11, fontweight='normal')
            ax.tick_params(axis='both', labelsize=8)
        else:
            ax.set_title(f"{title} (No Data)", fontsize=12, fontweight='normal')
            ax.set_xlabel(xlabel, fontsize=11)
            ax.set_ylabel('Frequency', fontsize=11)

    plt.tight_layout()
    output_png = root_path / "threshold_histograms.png"
    plt.savefig(output_png, dpi=300, bbox_inches='tight')
    print(f"\n直方图已保存至: {output_png}")
    plt.show()

if __name__ == "__main__":
    main()